# Video Retrieval Colab Setup

Self-contained notebook. Secrets come from **Colab Secrets**, then **Run all**.

Laptop UI stays separate:

```bash
video-index serve
```

Flow:

1. Add Colab Secrets + optional CONFIG defaults
2. Mount Drive
3. Clone / pull repo
4. Write `.env.colab`
5. Bootstrap ES + Qdrant (no keyframes)
6. Start worker `:8765`
7. Cloudflare tunnel → paste URL into laptop `.env`
8. Keepalive


## 0. CONFIG

Sensitive values come from **Colab Secrets** (left sidebar → key icon):

- `GEMINI_API_KEY` (required)
- `COLAB_REPO_URL` (required)
- `COLAB_REPO_BRANCH` (optional, default `cuong/use-colab`)
- `GITHUB_TOKEN` (optional, private repo)

Non-secret defaults are in the next cell; override only if needed.

In [ ]:
from google.colab import userdata

def secret(name: str, default: str = "") -> str:
    try:
        value = userdata.get(name)
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    return default

# Secrets from Colab Secrets sidebar (required / sensitive)
CONFIG = {
    "GEMINI_API_KEY": secret("GEMINI_API_KEY"),
    "COLAB_REPO_URL": secret("COLAB_REPO_URL"),
    "COLAB_REPO_BRANCH": secret("COLAB_REPO_BRANCH", "cuong/use-colab"),
    "GITHUB_TOKEN": secret("GITHUB_TOKEN"),

    # Non-secret defaults (override here if needed)
    "DRIVE_DATA_PATH": "MyDrive/video-retrieval",
    "QDRANT_COLLECTION": "video_keyframes_transnet",
    "ES_INDEX": "video_text_transnet",
    "GEMINI_MODEL": "gemini-2.0-flash",
    "QUERY_PLANNER": "gemini",
    "VISUAL_BACKEND": "real",
    "OCR_BACKEND": "mock",
    "ASR_BACKEND": "mock",
    "SIGLIP_MODEL_ID": "google/siglip-base-patch16-224",
    "BEIT3_MODEL_ID": "microsoft/beit-base-patch16-224",
    "SIGLIP_DIM": "768",
    "BEIT3_DIM": "768",
    "WHISPER_MODEL": "base",
    "DRIVE_MOUNT": "/content/drive",
    "ELASTICSEARCH_URL": "http://localhost:9200",
    "COLAB_ELASTICSEARCH_INSTALL_DIR": "/content/elasticsearch",
    "COLAB_QDRANT_INSTALL_DIR": "/content/qdrant",
    "COLAB_REMOTE_DATA_DIR": "/content/data",
    "COLAB_WORKER_PORT": "8765",
}

missing = [k for k in ("GEMINI_API_KEY", "COLAB_REPO_URL") if not CONFIG.get(k)]
if missing:
    raise SystemExit(
        "Missing Colab Secrets: "
        + ", ".join(missing)
        + ". Open the key icon in the left sidebar and add them."
    )

print("CONFIG ok (secrets loaded from Colab Secrets)")
print("repo=", CONFIG["COLAB_REPO_URL"])
print("branch=", CONFIG["COLAB_REPO_BRANCH"])
print("github_token_set=", bool(CONFIG["GITHUB_TOKEN"]))
print("drive=", CONFIG["DRIVE_DATA_PATH"])
print("qdrant=", CONFIG["QDRANT_COLLECTION"])
print("es=", CONFIG["ES_INDEX"])


## 1. Mount Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT = CONFIG["DRIVE_MOUNT"].rstrip("/") or "/content/drive"
drive.mount(MOUNT)

drive_root = Path(MOUNT) / CONFIG["DRIVE_DATA_PATH"].strip("/")
assert drive_root.is_dir(), f"Missing Drive folder: {drive_root}"
print("Drive OK:", drive_root)
print("Contents:", sorted(p.name for p in drive_root.iterdir()))

## 2. Clone / pull repo

In [ ]:
import subprocess
from pathlib import Path
from urllib.parse import urlparse, urlunparse

REPO = Path("/content/video-retrieval")
repo_url = CONFIG["COLAB_REPO_URL"].strip()
branch = CONFIG["COLAB_REPO_BRANCH"].strip() or "cuong/use-colab"
token = CONFIG.get("GITHUB_TOKEN", "").strip()

def auth_url(url: str) -> str:
    if not token or url.startswith("git@"):
        return url
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or parsed.username:
        return url
    netloc = f"{token}@{parsed.hostname}"
    if parsed.port:
        netloc += f":{parsed.port}"
    return urlunparse(parsed._replace(netloc=netloc))

auth = auth_url(repo_url)
if REPO.is_dir() and (REPO / ".git").is_dir():
    print(f"Updating existing repo at {REPO} (branch={branch})...")
    subprocess.check_call(["git", "remote", "set-url", "origin", auth], cwd=REPO)
    subprocess.check_call(["git", "fetch", "--depth", "1", "origin", branch], cwd=REPO)
    subprocess.check_call(["git", "checkout", "-B", branch, f"origin/{branch}"], cwd=REPO)
else:
    if REPO.exists():
        raise SystemExit(f"{REPO} exists but is not a git repo. Remove it first.")
    print(f"Cloning {repo_url} -> {REPO} (branch={branch})...")
    subprocess.check_call(
        ["git", "clone", "--branch", branch, "--depth", "1", auth, str(REPO)]
    )

print("Repo ready at", REPO)

## 3. Write `.env.colab` from CONFIG

In [ ]:
from pathlib import Path

env_path = Path("/content/video-retrieval/.env.colab")
env_keys = [
    "GEMINI_API_KEY",
    "GEMINI_MODEL",
    "QUERY_PLANNER",
    "VISUAL_BACKEND",
    "OCR_BACKEND",
    "ASR_BACKEND",
    "ELASTICSEARCH_URL",
    "ES_INDEX",
    "QDRANT_COLLECTION",
    "WHISPER_MODEL",
    "SIGLIP_MODEL_ID",
    "BEIT3_MODEL_ID",
    "SIGLIP_DIM",
    "BEIT3_DIM",
    "DRIVE_MOUNT",
    "DRIVE_DATA_PATH",
    "COLAB_ELASTICSEARCH_INSTALL_DIR",
    "COLAB_QDRANT_INSTALL_DIR",
    "COLAB_REMOTE_DATA_DIR",
    "COLAB_WORKER_PORT",
]

lines = ["# Generated by colab_setup.ipynb from CONFIG"]
for key in env_keys:
    value = str(CONFIG.get(key, "")).strip()
    if value:
        lines.append(f"{key}={value}")

env_path.parent.mkdir(parents=True, exist_ok=True)
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Wrote", env_path)


## 4. Bootstrap (Elasticsearch + Qdrant, no keyframes)

Pulls `elasticsearch/`, `qdrant/`, `manifests/` from Drive. Does **not** pull keyframes.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/video-retrieval")
os.chdir(REPO)
os.environ["COLAB_REMOTE_DATA_DIR"] = CONFIG["COLAB_REMOTE_DATA_DIR"]
os.environ["COLAB_WORKER_PORT"] = CONFIG["COLAB_WORKER_PORT"]
os.environ["PYTHONUNBUFFERED"] = "1"

# Stream child stdout/stderr live into this cell (avoid capture_output=True).
process = subprocess.Popen(
    ["python3", "-u", "scripts/colab/bootstrap_remote.py"],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
code = process.wait()
if code != 0:
    raise SystemExit(f"bootstrap_remote.py failed with exit code {code}")
print("Bootstrap done")


## 5. Start persistent worker on `:8765`

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/video-retrieval")
os.chdir(REPO)
os.environ["COLAB_REMOTE_DATA_DIR"] = CONFIG["COLAB_REMOTE_DATA_DIR"]
os.environ["COLAB_WORKER_PORT"] = CONFIG["COLAB_WORKER_PORT"]

subprocess.check_call(["python3", "scripts/colab/start_worker_remote.py"], cwd=REPO)
subprocess.check_call(["python3", "scripts/colab/worker_status_remote.py"], cwd=REPO)


## 5b. Cloudflare tunnel (laptop → worker)

Exposes the worker at a public `*.trycloudflare.com` URL so the laptop can call it **without** Colab CLI.

1. Run this cell (leave tunnel alive with keepalive).
2. Copy `COLAB_WORKER_PUBLIC_URL` into laptop `.env`.
3. On laptop: `video-index colab search "yellow lion"`


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/video-retrieval")
os.chdir(REPO)
os.environ["COLAB_REMOTE_DATA_DIR"] = CONFIG["COLAB_REMOTE_DATA_DIR"]
os.environ["COLAB_WORKER_PORT"] = CONFIG["COLAB_WORKER_PORT"]
os.environ["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    ["python3", "-u", "scripts/colab/start_tunnel_remote.py"],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
code = process.wait()
if code != 0:
    raise SystemExit(f"start_tunnel_remote.py failed with exit code {code}")

url_file = Path("/tmp/video-retrieval-worker-public-url.txt")
if url_file.is_file():
    print("\nPaste into laptop .env:")
    print(f"COLAB_WORKER_PUBLIC_URL={url_file.read_text().strip()}")
    print("COLAB_WORKER_MODE=tunnel")


## Laptop UI

After the **Cloudflare tunnel** cell prints a URL, put this in laptop `.env`:

```env
REMOTE_COMPUTE=colab
COLAB_WORKER_PUBLIC_URL=https://xxxx.trycloudflare.com
COLAB_WORKER_MODE=tunnel
```

Then:

```bash
video-index serve
# or
video-index colab search "yellow lion"
```

No live `colab` CLI session is required for search when `COLAB_WORKER_PUBLIC_URL` is set.


## 6. Keepalive

Leave this cell running. Do not run heavy laptop `colab exec` monitors while searching.

In [ ]:
import time
import urllib.request
from datetime import datetime

port = CONFIG["COLAB_WORKER_PORT"]
WORKER = f"http://127.0.0.1:{port}/health"
QDRANT = "http://127.0.0.1:6333/readyz"
ES = "http://127.0.0.1:9200/_cluster/health"
INTERVAL_SEC = 300  # 5 minutes

def ok(url: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            return 200 <= resp.status < 300
    except Exception:
        return False

print("Keepalive started. Leave this cell running.")
while True:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"[{now}] worker={ok(WORKER)} qdrant={ok(QDRANT)} es={ok(ES)}",
        flush=True,
    )
    time.sleep(INTERVAL_SEC)

## Optional: pull keyframes later

Only if you need VM-local keyframes:

```python
import subprocess
from pathlib import Path
subprocess.check_call(
    ["python3", "scripts/colab/pull_data_remote.py", "--with-keyframes"],
    cwd=Path("/content/video-retrieval"),
)
```